In [1]:
# Conv2d was removed from the final version of the paper


!pip install -q keras-flops psutil

import os
import time
import platform
import psutil
import numpy as np

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Conv1D, Conv2D, MaxPooling1D, MaxPooling2D,
    Dense, Dropout, GlobalAveragePooling1D, GlobalAveragePooling2D,
    Concatenate
)

# ============================================================
print("=" * 72)
print("  SYSTEM SPECIFICATIONS  (Google Colab)")
print("=" * 72)
print(f"  Platform        : {platform.system()} {platform.release()}")
print(f"  Architecture    : {platform.machine()}")
print(f"  Python          : {platform.python_version()}")
print(f"  TensorFlow      : {tf.__version__}")
print(f"  CPU Cores       : {os.cpu_count()} logical")
mem = psutil.virtual_memory()
print(f"  RAM Total       : {mem.total  / (1024**3):.2f} GB")
print(f"  RAM Available   : {mem.available / (1024**3):.2f} GB")
gpus = tf.config.list_physical_devices('GPU')
print(f"  GPU Available   : {'Yes' if gpus else 'No'}")
if gpus:
    try:
        props = tf.config.experimental.get_device_details(gpus[0])
        print(f"  GPU Name        : {props.get('device_name', 'N/A')}")
    except:
        print(f"  GPU Name        : {gpus[0].name}")
print("=" * 72)


# ============================================================
def make_conv2d_branch():
    inp = Input(shape=(240, 320, 3), name='input')
    x = Conv2D(32, (3,3), activation='relu')(inp)
    x = MaxPooling2D((2,2))(x)
    x = Conv2D(64, (3,3), activation='relu')(x)
    x = MaxPooling2D((2,2))(x)
    x = Conv2D(128, (3,3), activation='relu')(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.5)(x)
    out = Dense(12, activation='softmax', name='output')(x)
    return Model(inp, out)

def make_conv1d_branch():
    inp = Input(shape=(1250, 1), name='input')
    x = Conv1D(32, 3, activation='relu')(inp)
    x = MaxPooling1D(2)(x)
    x = Conv1D(64, 3, activation='relu')(x)
    x = MaxPooling1D(2)(x)
    x = Conv1D(128, 3, activation='relu')(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.5)(x)
    out = Dense(12, activation='softmax', name='output')(x)
    return Model(inp, out)

def make_conv1d_fast_branch():
    inp = Input(shape=(1250, 1), name='input')
    x = Conv1D(32, 3, activation='relu')(inp)
    x = MaxPooling1D(2)(x)
    x = Conv1D(64, 3, activation='relu')(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.5)(x)
    out = Dense(12, activation='softmax', name='output')(x)
    return Model(inp, out)

def make_conv1d_faster_branch():
    inp = Input(shape=(1250, 1), name='input')
    x = Conv1D(16, 3, activation='relu')(inp)
    x = MaxPooling1D(2)(x)
    x = Conv1D(32, 3, activation='relu')(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.5)(x)
    out = Dense(12, activation='softmax', name='output')(x)
    return Model(inp, out)

def make_fusion():
    """مدل فیوژن: Dense(16, relu) -> Dense(12, softmax)"""
    inp = Input(shape=(36,), name='fusion_input')
    x = Dense(16, activation='relu')(inp)
    out = Dense(12, activation='softmax')(x)
    return Model(inp, out)

def make_full_system(branch_builder, input_shape):
    """سیستم کامل: ۳ شاخه موازی + فیوژن"""
    in1 = Input(shape=input_shape, name='ch1')
    in2 = Input(shape=input_shape, name='ch2')
    in3 = Input(shape=input_shape, name='ch3')

    b1 = branch_builder()
    b2 = branch_builder()
    b3 = branch_builder()

    o1 = b1(in1)
    o2 = b2(in2)
    o3 = b3(in3)

    concat = Concatenate(name='concat')([o1, o2, o3])

    x = Dense(16, activation='relu')(concat)
    out = Dense(12, activation='softmax', name='final_output')(x)

    return Model([in1, in2, in3], out)


# ============================================================
def count_conv1d_flops(model):
    total = 0
    for layer in model.layers:
        cfg = layer.get_config()
        if isinstance(layer, Conv1D):
            in_shape  = layer.input.shape
            out_shape = layer.output.shape
            L_out, C_in, C_out = out_shape[1], in_shape[2], out_shape[2]
            k = cfg['kernel_size'][0]
            total += 2 * C_in * k * C_out * L_out
            if cfg.get('use_bias', True):
                total += C_out * L_out
        elif isinstance(layer, Dense):
            in_dim  = layer.input.shape[-1]
            out_dim = layer.output.shape[-1]
            total += 2 * in_dim * out_dim
            if cfg.get('use_bias', True):
                total += out_dim
        elif isinstance(layer, GlobalAveragePooling1D):
            s = layer.input.shape
            total += s[1] * s[2]
    return int(total)

def count_conv2d_flops(model):
    total = 0
    for layer in model.layers:
        cfg = layer.get_config()
        if isinstance(layer, Conv2D):
            in_shape  = layer.input.shape
            out_shape = layer.output.shape
            H_out, W_out = out_shape[1], out_shape[2]
            C_in, C_out  = in_shape[3], out_shape[2]
            kh, kw = cfg['kernel_size']
            total += 2 * C_in * kh * kw * C_out * H_out * W_out
            if cfg.get('use_bias', True):
                total += C_out * H_out * W_out
        elif isinstance(layer, Dense):
            in_dim  = layer.input.shape[-1]
            out_dim = layer.output.shape[-1]
            total += 2 * in_dim * out_dim
            if cfg.get('use_bias', True):
                total += out_dim
        elif isinstance(layer, GlobalAveragePooling2D):
            s = layer.input.shape
            total += s[1] * s[2] * s[3]
    return int(total)

def count_fusion_flops():
    f = (2*36*16 + 16) + 16 + (2*16*12 + 12) + 60
    return int(f)


# ============================================================
def measure_time(model, dummy_input, n_warmup=30, n_runs=200):
    with tf.device('/CPU:0'):
        for _ in range(n_warmup):
            _ = model.predict(dummy_input, verbose=0)
        times = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            _ = model.predict(dummy_input, verbose=0)
            times.append((time.perf_counter() - t0) * 1000.0)
    return float(np.mean(times)), float(np.std(times))


# ============================================================
CONFIGS = [
    ("conv2d",         make_conv2d_branch,         (240, 320, 3), True ),
    ("conv1d",         make_conv1d_branch,         (1250, 1),     False),
    ("conv1d_fast",    make_conv1d_fast_branch,    (1250, 1),     False),
    ("conv1d_faster",  make_conv1d_faster_branch,  (1250, 1),     False),
]

results = []

for name, builder, shape, is_2d in CONFIGS:
    print(f"\n{'─'*72}")
    print(f"  Profiling:  {name}")
    print(f"{'─'*72}")

    branch = builder()
    br_input_dummy = np.random.randn(1, *shape).astype(np.float32)
    _ = branch(br_input_dummy)

    br_params = branch.count_params()
    br_flops  = count_conv2d_flops(branch) if is_2d else count_conv1d_flops(branch)
    br_size   = br_params * 4 / 1024
    br_input  = np.random.randn(1, *shape).astype(np.float32)
    br_t, br_s = measure_time(branch, br_input)

    full = make_full_system(builder, shape)
    dummy_full_input = [np.random.randn(1, *shape).astype(np.float32) for _ in range(3)]
    _ = full(dummy_full_input)

    fu_params = full.count_params()
    fu_flops  = 3 * br_flops + count_fusion_flops()
    fu_size   = fu_params * 4 / 1024
    fu_input  = [br_input, br_input, br_input]
    fu_t, fu_s = measure_time(full, fu_input)

    results.append(dict(
        name=name,
        br_params=br_params,   br_flops=br_flops,
        br_size=br_size,       br_time=br_t,       br_std=br_s,
        fu_params=fu_params,   fu_flops=fu_flops,
        fu_size=fu_size,       fu_time=fu_t,       fu_std=fu_s,
    ))

    print(f"  Single branch : params={br_params:>10,}  FLOPs={br_flops:>12,}"
          f"  size={br_size:>8.2f} kB  time={br_t:.2f}±{br_s:.2f} ms")
    print(f"  Full system   : params={fu_params:>10,}  FLOPs={fu_flops:>12,}"
          f"  size={fu_size:>8.2f} kB  time={fu_t:.2f}±{fu_s:.2f} ms")


# ============================================================
print("\n\n" + "=" * 72)
print("  SUMMARY TABLE \u2014 FULL SYSTEM (3 branches + fusion)")
print("=" * 72)
hdr = f"{'Model':<14} {'Params':>10} {'Size (kB)':>11} {'FLOPs':>12} {'MFLOPs':>9} {'Time (ms)':>14}"
print(hdr)
print("-" * 72)
for r in results:
    print(f"{r['name']:<14} {r['fu_params']:>10,} {r['fu_size']:>11.1f}"
          f" {r['fu_flops']:>12,} {r['fu_flops']/1e6:>9.3f}"
          f" {r['fu_time']:>7.2f} \u00b1 {r['fu_std']:.2f}")


# ============================================================
print("\n\n" + "=" * 72)
print("  LaTeX TABLE (copy-paste ready)")
print("=" * 72)
print(r"""
\begin{table}[!htbp]
\centering
\caption{Computational complexity analysis.}
\begin{tabular}{lrrrrr}
\toprule
\textbf{Model} & \textbf{Params} & \textbf{Size} & \textbf{FLOPs} & \textbf{Time} & \textbf{Complexity} \\
\midrule""")

complexity_labels = {'conv2d': 'High', 'conv1d': 'Medium', 'conv1d_fast': 'Low', 'conv1d_faster': 'Very low'}
for r in results:
    p = f"{r['fu_params']:,}".replace(',', '{,}')
    s = f"{r['fu_size']:.1f}"
    f = f"{r['fu_flops']/1e6:.3f}"
    t = rf" {r['fu_time']:.2f} $\pm$ {r['fu_std']:.2f}"
    c = complexity_labels[r['name']]
    # Using raw string print for row to handle backslashes
    print(rf"{r['name'].replace('_', '\\_'):<16} & {p} & {s} & {f} & {t} & {c} \\\\")

print(r"""\bottomrule
\end{tabular}
\end{table}
""")

  SYSTEM SPECIFICATIONS  (Google Colab)
  Platform        : Linux 6.6.122+
  Architecture    : x86_64
  Python          : 3.13.15
  TensorFlow      : 2.20.0
  CPU Cores       : 2 logical
  RAM Total       : 12.67 GB
  RAM Available   : 11.28 GB
  GPU Available   : No

────────────────────────────────────────────────────────────────────────
  Profiling:  conv2d
────────────────────────────────────────────────────────────────────────
  Single branch : params=   114,284  FLOPs=3,361,273,937  size=  446.42 kB  time=131.36±35.26 ms
  Full system   : params=   343,648  FLOPs=10,083,823,451  size= 1342.38 kB  time=221.19±60.27 ms

────────────────────────────────────────────────────────────────────────
  Profiling:  conv1d
────────────────────────────────────────────────────────────────────────
  Single branch : params=    52,076  FLOPs=  23,271,468  size=  203.42 kB  time=96.42±29.12 ms
  Full system   : params=   157,024  FLOPs=  69,816,044  size=  613.38 kB  time=104.45±22.53 ms

─────────